# Track 2 — Smart Building (IoT / Energy)
### EP0418 · AI for Real-World Data and Automated Decisions · Starter Pack

**Who this suits:** engineering students (EEE, MAE, built-environment).
**Data:** 60 days of 5-minute classroom sensor readings — temperature, humidity,
CO₂, occupancy, energy.
**Macro lesson:** *Anomaly detection done well — contextual anomalies, stuck
sensors, drift.*
**Your built-in baseline:** fixed-threshold alerts.

---
> ### This dataset is deliberately broken
> It contains **six** distinct data-quality faults: a stuck sensor, a slow
> calibration drift, short equipment spikes, a missing block, duplicated rows,
> and dropouts encoded as a sentinel value rather than as blanks.
>
> Finding them is the assignment. Your first plot will show you maybe two.
> Your tutor has the answer key — don't ask for it until you've hunted.

Swap in real 5G & AIoT Lab data by pointing `CSV` at your own export; every cell
below works on any file with a `timestamp` column.

In [ ]:
import os, urllib.request
# Fetch this pack's data from the course repository if it is not here yet.
for _f in ['smart_building_sensors.csv']:
    if not os.path.exists(_f):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/starter_packs/track2_smart_building/" + _f, _f)

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import os

CSV = "smart_building_sensors.csv"
if not os.path.exists(CSV):
    from google.colab import files; files.upload()      # pick the CSV from the pack

df = pd.read_csv(CSV, parse_dates=["timestamp"])

In [ ]:
# --- FIRST LOOK -------------------------------------------------------
print("rows:", len(df), " columns:", list(df.columns))
print("range:", df.timestamp.min(), "->", df.timestamp.max())
print("\nblank (NaN) values:\n", df.isna().sum())
print("\nduplicated timestamps:", df.timestamp.duplicated().sum())
df.describe().T[["min", "mean", "max"]]

### Pause here and read that `describe()` table

One of those columns has a **minimum of -999**. That is not a real humidity
reading — it is how this logger records "sensor did not respond". pandas counted
it as a number, so it is silently dragging the mean down and it will not show up
in `isna()`. Loggers do this constantly in the real world (SLO 4.1).

How many other columns have a min or max that is physically impossible?

In [ ]:
# --- PLOT: all four signals over the full 60 days ---------------------
sig = ["temp_c", "humidity_pct", "co2_ppm", "energy_kwh"]
fig, axes = plt.subplots(len(sig), 1, figsize=(13, 9), sharex=True)
for ax, c in zip(axes, sig):
    ax.plot(df.timestamp, df[c], lw=.5)
    ax.set_ylabel(c); ax.grid(alpha=.3)
axes[0].set_title("Classroom sensors — 60 days, 5-minute interval")
plt.tight_layout(); plt.show()

In [ ]:
# --- YOUR BASELINE: FIXED-THRESHOLD ALERTS ----------------------------
# The dumbest possible alarm: fire whenever CO2 goes above 1000 ppm.
# This is what most real buildings actually run. Beat it.

ALERT = df.co2_ppm > 1000
print(f"FIXED-THRESHOLD BASELINE  (CO2 > 1000 ppm)")
print(f"  alerts fired : {ALERT.sum()} of {len(df)} readings ({ALERT.mean():.1%})")

# How many alerts fire when the room is empty? Those are false alarms.
empty = ALERT & (df.occupancy == 0)
print(f"  fired while room EMPTY : {empty.sum()}  <-- these are false alarms")

by_week = ALERT.groupby(df.timestamp.dt.isocalendar().week).sum()
print("\n  alerts per week:"); print(by_week.to_string())
print("\n>>> Notice the alert count climbing in the final weeks.")
print(">>> The room did not get worse. A sensor did. That is DRIFT (SLO 10.1),")
print(">>> and a fixed threshold cannot tell the difference. Yours should.")

---
## Three macro questions to investigate

**1. Is this reading unusual, or just unusual *right now*?**
28°C at 3 a.m. in an empty room is strange. 28°C at 2 p.m. with 30 people in the
room is Tuesday. An anomaly detector that ignores context will drown you in false
alarms. Compare a plain z-score against a z-score computed *within* each
hour-of-day-and-occupancy bucket (SLO 5.1, 5.3).

**2. How would you catch a sensor that stops reporting change?**
A stuck sensor does not produce outliers — it produces a flat line, which looks
beautifully well-behaved to a threshold alarm. What rolling statistic goes to
zero when a sensor freezes? (Hint: you are already asked to plot rolling standard
deviation in the SLO 3.4 lab note.)

**3. When your alert fires at 2 a.m., what should actually happen?**
An alert nobody reads is not a system (SLO 7.2, 7.3). Who is notified, through
what channel, and what makes it *actionable* rather than noise? Design the
escalation rule, then argue for your threshold — including what it costs you when
you set it too low.

## Now hand over to your AI assistant

Everything above was written for you. From here you drive. Open ChatGPT, Claude,
Copilot, or Cursor and work the **prompt → generate → verify → refine** loop
(SLO 9.1). A prompt worth copying looks like this:

> I have a pandas DataFrame `df` with columns `timestamp, temp_c, humidity_pct, co2_ppm, occupancy, energy_kwh`.
> The time column is `timestamp`. Replace the -999 sentinel values with NaN, drop duplicate timestamps, resample to a regular 5-minute grid, then flag anomalies using a rolling z-score computed within each hour-of-day bucket.
> Show me the code and explain what each step does.

**Then verify before you trust it (SLO 9.2).** Every time:

| Check | How |
|---|---|
| Does it run? | Run the cell. Read the error, don't just re-prompt. |
| Right shape? | `df.shape` before and after — did you lose rows you didn't mean to? |
| Right range? | `df.describe()` — any impossible values (negative rainfall, 300 bpm)? |
| Spot-check | Pick 2 rows by hand and confirm the number is what you'd compute yourself. |
| Beat baseline? | Compare against the baseline above. If it loses, say so honestly. |

**Keep a `verification_log.md`** as you go — one line per AI mistake you caught and
how you fixed it. That log is assessed in CA3, and an empty one reads as
"I did not check", not "the AI was perfect".